# Silver Layer

Clean raw data so useful metrics can be extracted in the next layer.

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from datetime import datetime

BRONZE_PATH = "/Volumes/workspace/default/bronze/aemo_raw"
SILVER_TABLE = "workspace.default.energy_clean"

BRONZE_SCHEMA = StructType([
    StructField("REGION", StringType(), True),
    StructField("SETTLEMENTDATE", StringType(), True),
    StructField("TOTALDEMAND", DoubleType(), True),
    StructField("RRP", DoubleType(), True),
    StructField("PERIODTYPE", StringType(), True)
])

def read_bronze():
    df = spark.read.csv(
        f"{BRONZE_PATH}/*.csv",
        header=True,
        schema=BRONZE_SCHEMA  # avoid infer schema
    )
    return df

def clean(df):
    df = df.filter(F.col("PERIODTYPE") == "TRADE")  # trade intervals only

    # parse settlement date
    df = df.withColumn(
        "settlement_ts",
        F.to_timestamp("SETTLEMENTDATE", "yyyy/MM/dd HH:mm:ss")
    )

    # cast types
    df = df.withColumn("total_demand_mw", F.col("TOTALDEMAND").cast(DoubleType()))
    df = df.withColumn("price_rrp", F.col("RRP").cast(DoubleType()))

    df = df.drop("SETTLEMENTDATE", "TOTALDEMAND", "RRP", "PERIODTYPE")  # drop raw columns

    df = df.dropDuplicates(["settlement_ts", "REGION"])  # drop overlapping times & regions

    df = df.withColumnRenamed("REGION", "region")  # rename after dedup

    df = df.dropna(subset=["settlement_ts", "total_demand_mw"])  # drop nulls

    df = df.filter(F.year("settlement_ts") < datetime.now().year)  # exclude Jan 1 this year

    return df

def write_silver(df):
    if spark.catalog.tableExists(SILVER_TABLE):
        # merge into existing managed table
        target = DeltaTable.forName(spark, SILVER_TABLE)
        (target.alias("t")
            .merge(df.alias("s"), "t.settlement_ts = s.settlement_ts AND t.region = s.region")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
        print("[SILVER] upsert complete")
    else:
        (df.write
            .format("delta")
            .saveAsTable(SILVER_TABLE))
        print("[SILVER] initial write complete")

def main():
    df_raw = read_bronze()
    print(f"[SILVER] raw row count: {df_raw.count()}")

    df_clean = clean(df_raw)
    print(f"[SILVER] clean row count: {df_clean.count()}")

    write_silver(df_clean)
    df_clean.printSchema()
    df_clean.show(5)

## Transformation

Run the test cell to verify a single batch before running `main` to process all files.

### Testing

In [0]:
df_raw = spark.read.csv(
    f"{BRONZE_PATH}/2024_01_VIC1.csv",
    header=True,
    inferSchema=True
)
print(f"raw rows: {df_raw.count()}")
df_raw.printSchema()
df_raw.show(3)

df_clean = clean(df_raw)
print(f"clean rows: {df_clean.count()}")
df_clean.printSchema()
df_clean.show(3)

### Running



In [0]:
main()

## Checks

### Date Verification

The silver layer should filter the data to exclude any records in the current year. The Dec 31 midnight reading is included in the Dec 31 file despite its timestamp being in Jan 1 of the next year.

In [0]:
df_silver = spark.read.table(SILVER_TABLE)

df_silver.agg(
    F.min("settlement_ts").alias("earliest"),
    F.max("settlement_ts").alias("latest")
).show()

df_silver.sort(F.desc("settlement_ts")).show(5)

### Removal

When the data is outside the expected date range, it should be removed to protect the integrity of the forecasting model.

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, SILVER_TABLE)
target.delete(F.year("settlement_ts") >= datetime.now().year)

spark.read.table(SILVER_TABLE).agg(
    F.max("settlement_ts").alias("latest")
).show()